[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_04/notebook_4_1_data_quality_assessment.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 4.1: Data Quality Assessment in Healthcare

**Chapter 4: Data Preparation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Assess data completeness, consistency, and validity
2. Detect outliers and anomalous values
3. Generate comprehensive data quality reports
4. Visualize data quality issues systematically
5. Understand how poor data quality impacts model performance

## Clinical Context

Real-world healthcare data is messy:
- Missing values (lab not ordered, equipment failure)
- Inconsistencies (different units, typos)
- Outliers (data entry errors, rare true values)
- Temporal issues (delayed documentation)

**"Garbage in, garbage out"** applies doubly in healthcare where bad predictions harm patients.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

## 1. Generate Realistic Messy Healthcare Data

In [ ]:
def generate_messy_ehr_data(n_patients=1000):
    """Generate realistic EHR data with quality issues."""
    data = []

    for i in range(n_patients):
        # Basic demographics
        age = np.random.normal(65, 15)
        age = max(18, min(100, age))

        # Introduce missing values (15%)
        if np.random.rand() < 0.15:
            age = np.nan

        sex = np.random.choice(['M', 'F', 'Male', 'Female', 'm', 'f', None],
                              p=[0.35, 0.35, 0.10, 0.10, 0.05, 0.03, 0.02])

        # Labs with missing values (30% missing)
        glucose = np.random.normal(120, 30) if np.random.rand() > 0.30 else np.nan

        # Introduce outliers (5%)
        if not np.isnan(glucose) and np.random.rand() < 0.05:
            glucose = np.random.choice([999, -1, 9999])  # Data entry errors

        # Blood pressure - sometimes recorded inconsistently
        if np.random.rand() > 0.20:
            sbp = np.random.normal(130, 20)
            # Different formats
            if np.random.rand() < 0.7:
                bp = f"{int(sbp)}/{int(sbp * 0.6)}"
            else:
                bp = int(sbp)  # Only systolic
        else:
            bp = None

        # Medications - inconsistent naming
        if np.random.rand() < 0.4:
            meds = np.random.choice(['Metformin', 'metformin', 'METFORMIN',
                                    'Metformin 500mg', None])
        else:
            meds = None

        data.append({
            'patient_id': f'P{i:04d}',
            'age': age,
            'sex': sex,
            'glucose': glucose,
            'bp': bp,
            'medications': meds
        })

    return pd.DataFrame(data)

df = generate_messy_ehr_data(1000)
print(f"Generated {len(df)} patient records")
df.head(10)

## 2. Data Quality Assessment

In [ ]:
class DataQualityAssessor:
    """Comprehensive data quality assessment."""

    def __init__(self, df):
        self.df = df
        self.report = {}

    def assess_completeness(self):
        """Check missing values."""
        missing = self.df.isnull().sum()
        missing_pct = (missing / len(self.df)) * 100

        self.report['completeness'] = pd.DataFrame({
            'missing_count': missing,
            'missing_pct': missing_pct
        }).sort_values('missing_pct', ascending=False)

        return self.report['completeness']

    def assess_consistency(self, column):
        """Check value consistency."""
        if column not in self.df.columns:
            return None

        values = self.df[column].dropna()
        value_counts = values.value_counts()

        # Check for similar values (e.g., 'M' vs 'Male')
        unique_values = set(str(v).lower() for v in values.unique())

        return {
            'unique_values': len(value_counts),
            'value_counts': value_counts,
            'normalized_unique': len(unique_values)
        }

    def detect_outliers(self, column, method='iqr', threshold=3):
        """Detect outliers in numeric columns."""
        if column not in self.df.columns:
            return None

        values = pd.to_numeric(self.df[column], errors='coerce').dropna()

        if method == 'iqr':
            Q1 = values.quantile(0.25)
            Q3 = values.quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers = (values < lower) | (values > upper)
        else:  # z-score
            z_scores = np.abs(stats.zscore(values))
            outliers = z_scores > threshold

        return {
            'n_outliers': outliers.sum(),
            'outlier_pct': (outliers.sum() / len(values)) * 100,
            'outlier_values': values[outliers].tolist()
        }

    def generate_report(self):
        """Generate comprehensive quality report."""
        print("="*80)
        print("DATA QUALITY ASSESSMENT REPORT")
        print("="*80)

        # Dataset overview
        print(f"\nDataset: {len(self.df)} rows, {len(self.df.columns)} columns")

        # Completeness
        print("\n1. COMPLETENESS")
        print("-" * 80)
        completeness = self.assess_completeness()
        print(completeness)

        # Consistency for categorical columns
        print("\n2. CONSISTENCY (Categorical)")
        print("-" * 80)
        for col in ['sex', 'medications']:
            if col in self.df.columns:
                consistency = self.assess_consistency(col)
                if consistency:
                    print(f"\n{col}:")
                    print(f"  Unique values: {consistency['unique_values']}")
                    print(f"  Top values:\n{consistency['value_counts'].head()}")

        # Outliers in numeric columns
        print("\n3. OUTLIERS (Numeric)")
        print("-" * 80)
        for col in ['age', 'glucose']:
            if col in self.df.columns:
                outliers = self.detect_outliers(col)
                if outliers:
                    print(f"\n{col}:")
                    print(f"  Outliers: {outliers['n_outliers']} ({outliers['outlier_pct']:.1f}%)")
                    if outliers['outlier_values']:
                        print(f"  Examples: {outliers['outlier_values'][:5]}")

        print("\n" + "="*80)

# Run assessment
assessor = DataQualityAssessor(df)
assessor.generate_report()

## 3. Visualize Data Quality Issues

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Missing value heatmap
ax = axes[0, 0]
missing_matrix = df.isnull()
sns.heatmap(missing_matrix.iloc[:100], cbar=False, yticklabels=False, ax=ax)
ax.set_title('Missing Data Pattern (First 100 rows)', fontweight='bold')

# Missing percentage by column
ax = axes[0, 1]
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct.plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Missing %')
ax.set_title('Missing Data by Column', fontweight='bold')
ax.grid(True, alpha=0.3)

# Glucose distribution with outliers
ax = axes[1, 0]
glucose_clean = pd.to_numeric(df['glucose'], errors='coerce').dropna()
ax.hist(glucose_clean[glucose_clean < 500], bins=50, alpha=0.7, color='steelblue')
ax.axvline(glucose_clean.median(), color='red', linestyle='--', label=f'Median: {glucose_clean.median():.0f}')
ax.set_xlabel('Glucose (mg/dL)')
ax.set_title('Glucose Distribution (excluding extreme outliers)', fontweight='bold')
ax.legend()

# Age box plot with outliers
ax = axes[1, 1]
age_clean = df['age'].dropna()
ax.boxplot(age_clean, vert=False)
ax.set_xlabel('Age (years)')
ax.set_title('Age Distribution (Box Plot)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Impact on Model Performance

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Create a synthetic outcome
df['outcome'] = ((df['age'].fillna(65) > 70) &
                (pd.to_numeric(df['glucose'], errors='coerce').fillna(120) > 140)).astype(int)

# Scenario 1: Use dirty data (simple imputation)
df_dirty = df.copy()
df_dirty['age'] = df_dirty['age'].fillna(df_dirty['age'].median())
df_dirty['glucose'] = pd.to_numeric(df_dirty['glucose'], errors='coerce').fillna(120)

X_dirty = df_dirty[['age', 'glucose']]
y = df_dirty['outcome']

X_train, X_test, y_train, y_test = train_test_split(X_dirty, y, test_size=0.3, random_state=42)

clf_dirty = RandomForestClassifier(n_estimators=100, random_state=42)
clf_dirty.fit(X_train, y_train)
y_pred_dirty = clf_dirty.predict(X_test)
acc_dirty = accuracy_score(y_test, y_pred_dirty)

# Scenario 2: Clean the data first
df_clean = df.copy()
# Remove extreme outliers
df_clean['glucose'] = pd.to_numeric(df_clean['glucose'], errors='coerce')
df_clean = df_clean[(df_clean['glucose'] < 500) | df_clean['glucose'].isna()]
# Better imputation (could use more sophisticated methods)
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['glucose'] = df_clean['glucose'].fillna(df_clean['glucose'].median())

X_clean = df_clean[['age', 'glucose']]
y_clean = df_clean['outcome']

X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.3, random_state=42)

clf_clean = RandomForestClassifier(n_estimators=100, random_state=42)
clf_clean.fit(X_train, y_train)
y_pred_clean = clf_clean.predict(X_test)
acc_clean = accuracy_score(y_test, y_pred_clean)

print("="*60)
print("IMPACT OF DATA QUALITY ON MODEL PERFORMANCE")
print("="*60)
print(f"Model trained on DIRTY data: {acc_dirty:.3f} accuracy")
print(f"Model trained on CLEAN data: {acc_clean:.3f} accuracy")
print(f"Improvement: {(acc_clean - acc_dirty):.3f}")
print("="*60)
print("\n💡 In this toy example both dirty and cleaned data led to the same results!")

## 5. Key Takeaways

### What We Learned

1. **Completeness**: Check missing value patterns
2. **Consistency**: Standardize formats and values
3. **Validity**: Detect outliers and impossible values
4. **Impact**: Poor data quality degrades model performance
5. **Process**: Always assess quality before modeling

### Real-World Considerations

- EHR data is inherently messy
- Missing data is not random (MNAR often)
- Outliers may be errors OR rare true values
- Documentation varies by provider
- Data quality assessment is ongoing

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 4: Data Preparation)*